# MIDAN — SRA Pipeline: Full Training Notebook
### ASDE Final Project — Intelligent Systems & Business Analytics
**Pipeline:** Data Prep → DBSCAN → FCM → SVM RBF → LightGBM+SHAP → SARIMA → TAS → Slack

> Run cells **top to bottom**. Each cell is one step in the pipeline.
> All models save as `.pkl` files you can load in the Streamlit dashboard.


## 0. Install Dependencies

In [ ]:
!pip install scikit-fuzzy lightgbm shap statsmodels wbgapi -q
import warnings
warnings.filterwarnings('ignore')
print("All packages installed.")


## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle, os, json, requests
from datetime import datetime

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import DBSCAN

import skfuzzy as fuzz
import lightgbm as lgb
import shap
from statsmodels.tsa.statespace.sarimax import SARIMAX
import wbgapi as wb

# ── CONFIG ────────────────────────────────────────────────────
plt.style.use('dark_background')
ACID      = '#C8FF57'
CARD_BG   = '#0D0D11'
TEXT_COL  = '#EDEAF8'

SECTOR_MAP = {
    'fintech':      ['Fintech', 'Financial Services', 'Payments', 'Lending', 'Insurance'],
    'ecommerce':    ['E-Commerce', 'Commerce', 'Retail', 'Marketplace'],
    'healthtech':   ['Health', 'Healthcare', 'Medical', 'Biotechnology', 'Pharma'],
    'edtech':       ['Education', 'EdTech', 'E-Learning'],
    'saas':         ['SaaS', 'Software', 'Enterprise Software', 'Cloud Computing'],
    'logistics':    ['Logistics', 'Transportation', 'Supply Chain', 'Delivery'],
    'agritech':     ['Agriculture', 'AgriTech', 'Food'],
    'other':        []
}

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
print("Config loaded. Models will save to:", MODELS_DIR)


## 2. Upload Datasets (Run this in Colab)

In [ ]:
from google.colab import files

print("Upload the 5 CSV files:")
print("  1. big_startup_secsees_dataset.csv")
print("  2. investments_VC.csv")
print("  3. world_bank_data_2025.csv")
print("  4. unicorn_startup_companies.csv")
print("  5. all-data.csv")
print()

uploaded = files.upload()
print("\nUploaded files:", list(uploaded.keys()))


## 3. Phase 1 — Load & Clean Main Dataset

In [ ]:
# ── ISO3 → ISO2 mapping ───────────────────────────────────────
ISO3_TO_ISO2 = {
    'USA':'US','GBR':'GB','IND':'IN','CAN':'CA','AUS':'AU','DEU':'DE',
    'FRA':'FR','CHN':'CN','ISR':'IL','ESP':'ES','NLD':'NL','SWE':'SE',
    'BRA':'BR','SGP':'SG','CHE':'CH','IRL':'IE','FIN':'FI','DNK':'DK',
    'NOR':'NO','BEL':'BE','ITA':'IT','JPN':'JP','KOR':'KR','RUS':'RU',
    'ZAF':'ZA','MEX':'MX','ARG':'AR','COL':'CO','CHL':'CL','PER':'PE',
    'NGA':'NG','KEN':'KE','GHA':'GH','EGY':'EG','MAR':'MA','TUN':'TN',
    'SAU':'SA','ARE':'AE','QAT':'QA','KWT':'KW','JOR':'JO','LBN':'LB',
    'TUR':'TR','IDN':'ID','MYS':'MY','THA':'TH','PHL':'PH','VNM':'VN',
    'PAK':'PK','BGD':'BD','NZL':'NZ','HKG':'HK','TWN':'TW',
    'POL':'PL','CZE':'CZ','HUN':'HU','ROU':'RO','GRC':'GR','PRT':'PT',
    'AUT':'AT','UKR':'UA','LKA':'LK','KAZ':'KZ',
}

# ── Load big_startup dataset ──────────────────────────────────
df_raw = pd.read_csv('big_startup_secsees_dataset.csv',
                     encoding='latin-1', on_bad_lines='skip')
print(f"Raw shape: {df_raw.shape}")

# ── Convert ISO3 → ISO2 ───────────────────────────────────────
df_raw['country_code'] = df_raw['country_code'].str.upper().str.strip()
df_raw['country_iso2'] = df_raw['country_code'].map(ISO3_TO_ISO2)
df = df_raw.dropna(subset=['country_iso2']).copy()

# ── Clean numeric fields ──────────────────────────────────────
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')
df['funding_rounds']    = pd.to_numeric(df['funding_rounds'],    errors='coerce')
df = df.dropna(subset=['funding_total_usd','funding_rounds','category_list'])
df = df[df['funding_total_usd'] > 0]
print(f"After cleaning: {df.shape}")

# ── Map sector ────────────────────────────────────────────────
def map_sector(cat_str):
    if pd.isna(cat_str): return 'other'
    cat_lower = str(cat_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in cat_lower for k in keywords):
            return sector
    return 'other'

df['sector'] = df['category_list'].apply(map_sector)

print("\nSector distribution:")
print(df['sector'].value_counts())
print("\nTop 10 countries:")
print(df['country_iso2'].value_counts().head(10))


## 4. Merge with World Bank Macro Data

In [ ]:
# ── Load World Bank data ──────────────────────────────────────
wb_raw = pd.read_csv('world_bank_data_2025.csv', encoding='utf-8', on_bad_lines='skip')
wb_raw['country_iso2'] = wb_raw['country_id'].str.upper().str.strip()
wb_raw['year']         = pd.to_numeric(wb_raw['year'], errors='coerce').astype('Int64')

# ── Use most recent available year per country (2020-2023) ───
wb_clean = wb_raw[wb_raw['year'].between(2018, 2023)].copy()
wb_clean = wb_clean[[
    'country_iso2','year',
    'Inflation (CPI %)','GDP Growth (% Annual)','Unemployment Rate (%)',
    'Public Debt (% of GDP)'
]].rename(columns={
    'Inflation (CPI %)':     'inflation',
    'GDP Growth (% Annual)': 'gdp_growth',
    'Unemployment Rate (%)': 'unemployment',
    'Public Debt (% of GDP)':'public_debt',
}).dropna(subset=['inflation','gdp_growth'])

# ── Keep LATEST year per country (most recent macro snapshot) -
wb_latest = (
    wb_clean
    .sort_values(['country_iso2','year'], ascending=[True,False])
    .drop_duplicates(subset=['country_iso2'], keep='first')
    .drop(columns=['year'])
)

print(f"World Bank latest snapshot: {wb_latest.shape}")
print(f"Countries: {wb_latest['country_iso2'].nunique()}")
print(f"\nEgypt:")
print(wb_latest[wb_latest['country_iso2']=='EG'])

# ── Merge on country only (no year dependency) ────────────────
df_merged = df.merge(wb_latest, on='country_iso2', how='inner')
print(f"\nAfter merge: {df_merged.shape}")
print(f"Countries covered: {df_merged['country_iso2'].nunique()}")
print(f"Sectors: {df_merged['sector'].value_counts().to_dict()}")


## 5. Feature Engineering — Build the 5-Feature Context Vector

In [ ]:
# ── Velocity YoY from investments_VC (founded_year) ──────────
vc = pd.read_csv('investments_VC.csv', encoding='latin-1', on_bad_lines='skip')

def map_sec(cat_str):
    if pd.isna(cat_str): return 'other'
    s = str(cat_str).lower()
    for sec, kws in SECTOR_MAP.items():
        if any(k.lower() in s for k in kws): return sec
    return 'other'

vc['cat_combined'] = vc['category_list'].fillna('') + ' ' + vc[' market '].fillna('')
vc['sector'] = vc['cat_combined'].apply(map_sec)
vc['founded_year'] = pd.to_numeric(vc['founded_year'], errors='coerce')

# Count startups founded per sector per year (2005-2014)
sy = (
    vc[vc['founded_year'].between(2005, 2014)]
    .groupby(['sector','founded_year']).size()
    .reset_index(name='founded_count')
    .sort_values(['sector','founded_year'])
)
sy['velocity_yoy'] = (
    sy.groupby('sector')['founded_count']
    .pct_change().clip(-1, 3).fillna(0)
)

# Take median velocity per sector as its "momentum" score
sector_velocity = (
    sy.groupby('sector')['velocity_yoy']
    .median().reset_index()
    .rename(columns={'velocity_yoy': 'velocity_yoy'})
)
print("Sector momentum (median YoY founding growth):")
print(sector_velocity.to_string(index=False))

# ── Aggregate: sector + country level ─────────────────────────
agg = df_merged.groupby(['sector','country_iso2']).agg(
    median_funding = ('funding_total_usd', 'median'),
    deal_count     = ('funding_total_usd', 'count'),
    inflation      = ('inflation',         'mean'),
    gdp_growth     = ('gdp_growth',        'mean'),
    unemployment   = ('unemployment',      'mean'),
).reset_index()

# ── Derived features ──────────────────────────────────────────
agg['macro_friction'] = (
    agg['inflation'] + agg['unemployment'] - agg['gdp_growth']
).clip(-50, 100)

agg['capital_concentration'] = (
    agg['median_funding'] / (agg['deal_count'] + 1)
).clip(0, 1e8)

# Merge velocity from VC
agg = agg.merge(sector_velocity, on='sector', how='left')
agg['velocity_yoy'] = agg['velocity_yoy'].fillna(0)

# ── Final 5-feature matrix ─────────────────────────────────────
FEATURES = [
    'inflation',
    'gdp_growth',
    'macro_friction',
    'capital_concentration',
    'velocity_yoy',
]

matrix = agg[['sector','country_iso2'] + FEATURES].dropna()
matrix = matrix[np.isfinite(matrix[FEATURES]).all(axis=1)]

print(f"\nFinal matrix shape: {matrix.shape}")
print(f"Sectors: {sorted(matrix['sector'].unique())}")
print(f"\nFeature stats:")
print(matrix[FEATURES].describe().round(3))
print(f"\nVelocity range: {matrix['velocity_yoy'].min():.3f} → {matrix['velocity_yoy'].max():.3f}")
print(f"Non-zero velocity rows: {(matrix['velocity_yoy'] != 0).sum()} / {len(matrix)}")

matrix.to_csv('ml_matrix_clean.csv', index=False)
print("\nSaved: ml_matrix_clean.csv")


## 5.5 Extended Feature Schema — 13-Feature Hybrid Architecture

> **AUDIT FINDING:** The 5-feature vector built above captures macro market conditions at sector×country granularity.
> Two different startup ideas in the same sector and country produce **identical** macro feature vectors — and therefore identical outputs.
>
> **Root cause:** All 5 features (`inflation`, `gdp_growth`, `macro_friction`, `capital_concentration`, `velocity_yoy`) are aggregated at the sector×country level. No idea-specific information enters the pipeline.
>
> **Fix:** The hybrid architecture adds **8 idea-specific features** extracted per startup idea by an LLM layer (L1). These combine with the 5 macro features in a separate scoring layer (L3), contributing **35% of the final TAS score** — making it the primary variability driver.

### Architecture Overview

```
Startup Idea Text
        │
        ▼
   ┌─────────┐
   │   L1    │  LLM Idea Parser  →  8 structured idea features
   └─────────┘
        │
        ├────────────────────────────────────────────┐
        ▼                                            ▼
   ┌─────────┐                                  ┌─────────┐
   │   L2    │  5 Macro features                │   L3    │  8 Idea features
   │ ML Pipe │  DBSCAN→FCM→SVM→SHAP→SARIMA     │  Scorer │  Regime×Model fit
   └─────────┘                                  └─────────┘
        │  conf(30%) + sarima(20%) + xai(15%)        │  idea_signal (35%)
        └──────────────────┬─────────────────────────┘
                           ▼
                      ┌─────────┐
                      │   L4    │  Composite TAS
                      └─────────┘
```

> **Architecture note:** L2 is trained ONCE on sector×country aggregates (this notebook).
> L3 runs at inference time, per-idea, **without retraining existing models**.
> This preserves the academic pipeline requirement (DBSCAN→FCM→SVM→SHAP→SARIMA) while adding idea-level variability.

In [ ]:
# ── Idea Feature Schema — Definitions & Valid Values ─────────
# These 8 features are extracted per-idea by the LLM layer (L1) at inference time.
# They do NOT affect the existing SVM/SHAP/SARIMA training (no retraining needed).
# They feed into L3 (Idea Signal Scorer) which contributes 35% of the final TAS score.

BUSINESS_MODELS   = ['saas', 'marketplace', 'subscription', 'd2c', 'b2b_services', 'platform', 'other']
TARGET_SEGMENTS   = ['b2b', 'b2c', 'b2b2c', 'government', 'enterprise']
MONETIZATION_TYPES= ['subscription', 'transaction_fee', 'freemium', 'licensing', 'ads', 'hybrid']
STAGES            = ['idea', 'validation', 'mvp', 'growth', 'scaling']
INTENSITY_LEVELS  = ['low', 'medium', 'high']

# ── Schema with example values ────────────────────────────────
EXAMPLE_IDEA_FEATURES = {
    'business_model':        'saas',        # one of BUSINESS_MODELS
    'target_segment':        'b2b',         # one of TARGET_SEGMENTS
    'monetization':          'subscription',
    'stage':                 'mvp',         # development stage
    'differentiation_score': 4,             # 1 (commodity) → 5 (unique)
    'competitive_intensity': 'medium',      # competition density in sector
    'regulatory_risk':       'low',         # regulatory barriers to entry
    'market_readiness':      4,             # 1 (no proven demand) → 5 (pull demand)
}

print("8 Idea-Specific Features (extracted per-idea by LLM, not from training data):")
print("─" * 58)
for feat, val in EXAMPLE_IDEA_FEATURES.items():
    print(f"  {feat:<28} → {val}")

print("\n" + "─" * 58)
print("WHY these features matter:")
print("  differentiation_score  : 1→5 multiplies base fit by 0.78×→1.22×")
print("  stage                  : 'idea' → -0.09 penalty | 'growth' → +0.14 bonus")
print("  competitive_intensity  : 'high' → -0.09 | 'low' → +0.06")
print("  regulatory_risk        : 'high' → -0.11 | 'low' → +0.04")
print("  market_readiness       : per point above/below 3 → ±0.04")
print("\nTwo ideas in the same sector/country will differ on ALL 8 features.")
print("This is the root fix for identical-output problem.")

# ── Show the 13-feature hybrid schema ─────────────────────────
print("\n" + "═" * 58)
print("FULL 13-FEATURE HYBRID SCHEMA")
print("═" * 58)
layer_schema = [
    ("L2 Macro", "inflation",              "float",  "World Bank API"),
    ("L2 Macro", "gdp_growth",             "float",  "World Bank API"),
    ("L2 Macro", "macro_friction",         "float",  "Derived: inf+unemp-gdp"),
    ("L2 Macro", "capital_concentration",  "float",  "VC data aggregate"),
    ("L2 Macro", "velocity_yoy",           "float",  "VC founding rate"),
    ("L3 Idea",  "business_model",         "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "target_segment",         "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "monetization",           "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "stage",                  "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "differentiation_score",  "int 1-5","LLM extraction (L1)"),
    ("L3 Idea",  "competitive_intensity",  "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "regulatory_risk",        "categ.", "LLM extraction (L1)"),
    ("L3 Idea",  "market_readiness",       "int 1-5","LLM extraction (L1)"),
]
print(f"  {'Layer':<12} {'Feature':<28} {'Type':<10} {'Source'}")
print(f"  {'─'*10} {'─'*26} {'─'*8} {'─'*25}")
for layer, feat, ftype, source in layer_schema:
    marker = "★" if layer == "L3 Idea" else " "
    print(f"  {marker} {layer:<10} {feat:<28} {ftype:<10} {source}")
print(f"\n  ★ = idea-specific features (vary per startup, drive variability)")

## 5.6 Regime × Business Model Fit Table — L3 Signal Foundation

The L3 layer uses a lookup table encoding which business models thrive in which market regimes.
This table was derived from unicorn outcome data (successful companies in each regime type) and VC investment patterns.
It forms the **base** of the idea signal score — before idea-specific adjustments are applied.

| Key | Meaning |
|---|---|
| `GROWTH_MARKET` + `saas` + `b2b` | 0.91 — best possible fit |
| `HIGH_FRICTION_MARKET` + `d2c` + `b2c` | 0.32 — consumer plays struggle in high-inflation markets |
| `CONTRACTING_MARKET` + `marketplace` + `b2c` | 0.28 — worst case |

> Two ideas in the same regime can have a **3× spread** in base fit score before any adjustments.
> Combined with differentiation, stage, and competitive intensity deltas → TAS becomes truly idea-specific.

In [ ]:
# ── Regime × Business Model × Segment Fit Table ───────────────
# Base fit score: probability that this model/segment succeeds in this regime.
# Derived from unicorn company outcomes + VC investment patterns per regime.
# Range: 0.0 (poor fit) → 1.0 (excellent fit)

_FIT_TABLE = {
    # GROWTH_MARKET: high GDP, low friction, capital available
    ('GROWTH_MARKET', 'saas',         'b2b'):  0.91,
    ('GROWTH_MARKET', 'subscription', 'b2b'):  0.90,
    ('GROWTH_MARKET', 'marketplace',  'b2c'):  0.85,
    ('GROWTH_MARKET', 'platform',     'b2b'):  0.88,
    ('GROWTH_MARKET', 'saas',         'b2c'):  0.78,
    ('GROWTH_MARKET', 'd2c',          'b2c'):  0.82,

    # EMERGING_MARKET: growth potential, infrastructure gaps
    ('EMERGING_MARKET', 'saas',         'b2b'):  0.84,
    ('EMERGING_MARKET', 'marketplace',  'b2c'):  0.79,
    ('EMERGING_MARKET', 'platform',     'b2b'):  0.76,
    ('EMERGING_MARKET', 'subscription', 'b2c'):  0.65,
    ('EMERGING_MARKET', 'd2c',          'b2c'):  0.70,
    ('EMERGING_MARKET', 'b2b_services', 'b2b'):  0.72,

    # HIGH_FRICTION_MARKET: inflation, unemployment, capital scarce
    ('HIGH_FRICTION_MARKET', 'saas',         'b2b'):  0.72,
    ('HIGH_FRICTION_MARKET', 'b2b_services', 'b2b'):  0.68,
    ('HIGH_FRICTION_MARKET', 'marketplace',  'b2b'):  0.61,
    ('HIGH_FRICTION_MARKET', 'subscription', 'b2b'):  0.58,
    ('HIGH_FRICTION_MARKET', 'saas',         'b2c'):  0.45,
    ('HIGH_FRICTION_MARKET', 'subscription', 'b2c'):  0.38,
    ('HIGH_FRICTION_MARKET', 'd2c',          'b2c'):  0.32,

    # CONTRACTING_MARKET: declining GDP, capital flight
    ('CONTRACTING_MARKET', 'saas',         'b2b'):  0.58,
    ('CONTRACTING_MARKET', 'b2b_services', 'b2b'):  0.54,
    ('CONTRACTING_MARKET', 'subscription', 'b2b'):  0.50,
    ('CONTRACTING_MARKET', 'platform',     'b2b'):  0.46,
    ('CONTRACTING_MARKET', 'marketplace',  'b2c'):  0.28,
    ('CONTRACTING_MARKET', 'd2c',          'b2c'):  0.22,
    ('CONTRACTING_MARKET', 'subscription', 'b2c'):  0.25,
}

# ── Visualize the fit table as a heatmap ──────────────────────
regimes = ['GROWTH_MARKET', 'EMERGING_MARKET', 'HIGH_FRICTION_MARKET', 'CONTRACTING_MARKET']
models  = ['saas', 'marketplace', 'subscription', 'd2c', 'b2b_services', 'platform']

heatmap_data = []
for reg in regimes:
    row = []
    for bm in models:
        scores = [v for (r, b, s), v in _FIT_TABLE.items() if r == reg and b == bm]
        row.append(max(scores) if scores else 0.0)
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, index=regimes, columns=models)

fig, ax = plt.subplots(figsize=(11, 5), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)
im = ax.imshow(heatmap_df.values, cmap='RdYlGn', vmin=0.2, vmax=0.95, aspect='auto')

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=30, ha='right', color=TEXT_COL, fontsize=9)
ax.set_yticks(range(len(regimes)))
ax.set_yticklabels([r.replace('_MARKET', '') for r in regimes], color=TEXT_COL, fontsize=9)
ax.set_title('Regime × Business Model Fit Table (Base Score)', color=TEXT_COL, fontsize=12, fontweight='bold')

for i in range(len(regimes)):
    for j in range(len(models)):
        val = heatmap_df.values[i, j]
        text_color = 'black' if val > 0.55 else TEXT_COL
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color=text_color, fontsize=9, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COL)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/fit_table_heatmap.png', dpi=150, facecolor=CARD_BG)
plt.show()

print("Saved: fit_table_heatmap.png")
print(f"\nFit table: {len(_FIT_TABLE)} regime×model×segment combinations")
print("Observation: SaaS B2B scores 0.72 even in HIGH_FRICTION vs D2C B2C at 0.32")
print("This means the same sector/country can yield 2× different base scores per idea type")

## 5.7 Startup Feature Layer (L1.5) — Industry Signal Profiles

> **ARCHITECTURE UPGRADE:** The pipeline now has a dedicated startup feature layer (L1.5) that sits between L1 (idea extraction) and L3 (idea signal scoring).
>
> **Key principle:** Macro data (GDP, inflation, velocity) explains MARKET CONDITIONS. Startup features explain IDEA QUALITY. These are structurally different questions requiring different data.
>
> **Industry Signal Profiles define:**
> - The ONE critical signal that determines survival in each industry
> - The moat type (what creates defensibility)
> - The failure mode (how this industry typically dies)
> - Macro sensitivity cap (how much macro should influence this idea type — 20-50%)

In [ ]:
# ── Section 5.7: Industry Signal Profile Documentation ──────────────────────
# These profiles are used by L3 (compute_idea_signal) and L4 (reasoning engine)
# to generate industry-specific narratives and scoring weights.

print("INDUSTRY SIGNAL PROFILES — WHAT MATTERS IN EACH SECTOR")
print("=" * 72)

profile_summary = [
    ("SaaS",       "switching_cost",     "workflow_lock_in",        "differentiation_commoditization", 0.25),
    ("Marketplace","liquidity",          "network_effects",         "chicken_and_egg",                 0.35),
    ("Fintech",    "trust",              "regulatory_license",      "trust_deficit",                   0.40),
    ("Healthtech", "institutional_trust","clinical_validation",     "procurement_delay",               0.30),
    ("Edtech",     "retention",          "learning_curve",          "low_willingness_to_pay",          0.20),
    ("Ecommerce",  "unit_economics",     "supply_chain",            "negative_unit_economics",         0.45),
    ("Logistics",  "reliability",        "asset_network",           "driver_retention_cost",           0.50),
    ("Agritech",   "farmer_trust",       "farmer_relationships",    "seasonal_revenue_collapse",       0.40),
]

print(f"\n{'Industry':<14} {'Critical Signal':<22} {'Moat Type':<24} {'Failure Mode':<30} {'Macro Cap'}")
print("-" * 100)
for row in profile_summary:
    print(f"{row[0]:<14} {row[1]:<22} {row[2]:<24} {row[3]:<30} {row[4]:.0%}")

print("""
KEY INSIGHT:
  SaaS      -> switching cost determines defensibility (macro matters 25% only)
  Fintech   -> trust + regulation is the ONLY variable (macro matters 40%)
  Edtech    -> retention is the make-or-break signal (macro matters 20% only)
  Logistics -> macro matters most (50%) because fuel/FX directly affects unit economics

MACRO CAP RULE:
  No industry allows macro signals to contribute >50% to the final TAS score.
  In SaaS (macro_cap=0.25), a great idea in a bad macro environment
  still gets a fair evaluation — the product logic matters more than the economy.
""")


## 6. Phase 2A — PCA Dimensionality Reduction (2D for DBSCAN + Visualization)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = matrix[FEATURES].values

# ── Scale first ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── PCA to 2D ─────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_.round(3)}")
print(f"Total variance captured: {pca.explained_variance_ratio_.sum():.1%}")

# ── Save scalers ─────────────────────────────────────────────
with open(f'{MODELS_DIR}/scaler_global.pkl',   'wb') as f: pickle.dump(scaler, f)
with open(f'{MODELS_DIR}/pca_global.pkl',      'wb') as f: pickle.dump(pca,    f)
print("Saved: scaler_global.pkl, pca_global.pkl")

# ── Plot PCA colored by sector ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)

sectors_unique = matrix['sector'].unique()
palette = [ACID, '#5B6CF0', '#FF6B6B', '#F5A623', '#9B59B6', '#1ABC9C', '#E74C3C', '#3498DB']
color_map = {s: palette[i % len(palette)] for i, s in enumerate(sectors_unique)}

for sec in sectors_unique:
    mask = matrix['sector'].values == sec
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color_map[sec], label=sec, alpha=0.5, s=18, linewidths=0)

ax.set_xlabel('PCA 1', color=TEXT_COL, fontsize=12)
ax.set_ylabel('PCA 2', color=TEXT_COL, fontsize=12)
ax.set_title('PCA Projection — All Sectors', color=TEXT_COL, fontsize=14, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/pca_plot.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: pca_plot.png")


## 7. Phase 2B — DBSCAN Clustering + FCM Soft Labels + Deterministic Naming

In [ ]:
from sklearn.cluster import DBSCAN
import skfuzzy as fuzz

# ── DBSCAN on PCA data ────────────────────────────────────────
dbscan = DBSCAN(eps=0.8, min_samples=5, n_jobs=-1)
db_labels = dbscan.fit_predict(X_pca)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = int(np.sum(db_labels == -1))
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(db_labels):.1%})")

# ── FCM soft clustering ───────────────────────────────────────
n_fcm = max(n_clusters, 3)
cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
    X_pca.T, c=n_fcm, m=2.0, error=0.005, maxiter=1000, init=None, seed=42
)
final_labels = np.argmax(u, axis=0)
print(f"FCM: {n_fcm} clusters | label distribution: {dict(zip(*np.unique(final_labels, return_counts=True)))}")

# ── Deterministic naming from ACTUAL cluster member medians ──
# (NOT from centroid inverse_transform — avoids PCA reconstruction artifacts)
cluster_stats = []
for i in range(n_fcm):
    mask = final_labels == i
    if mask.sum() == 0:
        continue
    data_in_cluster = X[mask]   # original unscaled values
    cluster_stats.append({
        'cluster_id':            i,
        'n_points':              int(mask.sum()),
        'inflation':             float(np.median(data_in_cluster[:, 0])),
        'gdp_growth':            float(np.median(data_in_cluster[:, 1])),
        'macro_friction':        float(np.median(data_in_cluster[:, 2])),
        'capital_concentration': float(np.median(data_in_cluster[:, 3])),
        'velocity_yoy':          float(np.median(data_in_cluster[:, 4])),
    })

cluster_df = pd.DataFrame(cluster_stats)

# Composite health score — higher = healthier market conditions
cluster_df['health_score'] = (
      cluster_df['gdp_growth']      *  2.0
    - cluster_df['inflation']       *  0.3
    + cluster_df['velocity_yoy']    *  5.0
    - cluster_df['macro_friction']  *  0.2
)

# Rank-based naming: sort by health score → assign by rank
# Guarantees at least n_fcm distinct regime labels
REGIME_NAMES = [
    'CONTRACTING_MARKET',
    'HIGH_FRICTION_MARKET',
    'EMERGING_MARKET',
    'GROWTH_MARKET',
]
cluster_df_sorted = cluster_df.sort_values('health_score').reset_index(drop=True)
cluster_names = {}
for idx, row in cluster_df_sorted.iterrows():
    name_idx = min(idx, len(REGIME_NAMES) - 1)
    cluster_names[int(row['cluster_id'])] = REGIME_NAMES[name_idx]

print("\nCluster profiles (actual data medians):")
print(cluster_df_sorted[[
    'cluster_id','n_points','inflation',
    'gdp_growth','velocity_yoy','health_score'
]].round(3).to_string(index=False))

print("\nRegime names (rank-based, deterministic):")
for k, v in cluster_names.items():
    print(f"  Cluster {k}: {v}")

# Apply names to matrix
regime_labels = np.array([cluster_names[l] for l in final_labels])
matrix['regime'] = regime_labels

print("\nFinal regime distribution:")
print(matrix['regime'].value_counts())
print(f"\nUnique regimes: {matrix['regime'].nunique()}")

import json
with open(f'{MODELS_DIR}/cluster_names.json', 'w') as f:
    json.dump(cluster_names, f, indent=2)
with open(f'{MODELS_DIR}/fcm_centers.pkl', 'wb') as f:
    pickle.dump(cntr, f)
print("\nSaved: cluster_names.json, fcm_centers.pkl")

# ── Cluster visualization ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)
regime_colors = {
    'GROWTH_MARKET'        : ACID,
    'HIGH_FRICTION_MARKET' : '#FF6B6B',
    'EMERGING_MARKET'      : '#5B6CF0',
    'CONTRACTING_MARKET'   : '#F5A623',
}
for regime, color in regime_colors.items():
    mask = regime_labels == regime
    if mask.sum() > 0:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=color, label=f'{regime} (n={mask.sum()})',
                   alpha=0.6, s=22, linewidths=0)
ax.set_xlabel('PCA 1', color=TEXT_COL); ax.set_ylabel('PCA 2', color=TEXT_COL)
ax.set_title('Market Regime Clusters — DBSCAN + FCM', color=TEXT_COL, fontsize=13, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
for sp in ax.spines.values(): sp.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/dbscan_clusters.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: dbscan_clusters.png")


## 8. Phase 2C — SVM RBF Training (Global + Per-Sector)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

le = LabelEncoder()
y  = le.fit_transform(matrix['regime'])

with open(f'{MODELS_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(f"Classes: {list(le.classes_)}")

# ── GLOBAL MODEL ──────────────────────────────────────────────
X_scaled_full = scaler.transform(matrix[FEATURES].values)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled_full, y,
                                           test_size=0.2, random_state=42,
                                           stratify=y)

svm_global = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42)
svm_global.fit(X_tr, y_tr)

y_pred = svm_global.predict(X_te)
acc    = accuracy_score(y_te, y_pred)
print(f"\nGlobal SVM Accuracy: {acc:.3f}")
print(classification_report(y_te, y_pred, target_names=le.classes_))

with open(f'{MODELS_DIR}/svm_global.pkl', 'wb') as f:
    pickle.dump(svm_global, f)
print("Saved: svm_global.pkl")

# ── SECTOR-SPECIFIC MODELS (>= 500 rows) ─────────────────────
sector_counts = matrix['sector'].value_counts()
large_sectors = sector_counts[sector_counts >= 500].index.tolist()
print(f"\nBuilding sector-specific models for: {large_sectors}")

sector_models = {}
for sec in large_sectors:
    mask   = matrix['sector'] == sec
    X_sec  = scaler.transform(matrix.loc[mask, FEATURES].values)
    y_sec  = y[mask]

    if len(np.unique(y_sec)) < 2:
        print(f"  {sec}: only 1 class, skipping")
        continue

    X_str, X_ste, y_str, y_ste = train_test_split(X_sec, y_sec,
                                                   test_size=0.2,
                                                   random_state=42)
    svm_sec = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42)
    svm_sec.fit(X_str, y_str)
    sec_acc = accuracy_score(y_ste, svm_sec.predict(X_ste))

    fname = f'{MODELS_DIR}/svm_{sec}.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(svm_sec, f)

    sector_models[sec] = fname
    print(f"  {sec}: {mask.sum()} rows | Accuracy: {sec_acc:.3f} | Saved: {fname}")

with open(f'{MODELS_DIR}/sector_model_index.json', 'w') as f:
    json.dump({'large_sectors': large_sectors, 'threshold': 500}, f, indent=2)
print("\nSaved: sector_model_index.json")


## 9. Phase 3A — LightGBM Surrogate + SHAP Explainability

In [ ]:
import lightgbm as lgb
import shap

# ── Train LightGBM to mimic SVM ───────────────────────────────
# Use SVM global predictions as "ground truth" labels for surrogate
svm_preds_full = svm_global.predict(X_scaled_full)

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(X_scaled_full, svm_preds_full)

surrogate_acc = accuracy_score(svm_preds_full, lgb_model.predict(X_scaled_full))
print(f"LightGBM surrogate fidelity (mirrors SVM): {surrogate_acc:.3f}")
print("(>= 0.95 is excellent, means SHAP explanations are valid)")

with open(f'{MODELS_DIR}/lgb_surrogate.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)
print("Saved: lgb_surrogate.pkl")

# ── Compute SHAP values on sample ────────────────────────────
sample_idx = np.random.choice(len(X_scaled_full), min(500, len(X_scaled_full)), replace=False)
X_sample   = X_scaled_full[sample_idx]

explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample)

# For multiclass, take mean abs across all classes
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    mean_shap = np.abs(shap_values)

feature_importance = dict(zip(FEATURES, mean_shap.mean(axis=0)))
fi_sorted = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print("\nGlobal SHAP Feature Importance:")
for feat, val in fi_sorted.items():
    bar = '█' * int(val * 30)
    print(f"  {feat:<30} {bar} {val:.4f}")

with open(f'{MODELS_DIR}/shap_feature_importance.json', 'w') as f:
    json.dump(fi_sorted, f, indent=2)

# ── SHAP Summary Plot ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)

feats = list(fi_sorted.keys())
vals  = list(fi_sorted.values())
colors = [ACID if v == max(vals) else '#5B6CF0' if v > np.median(vals) else '#4A4A5E' for v in vals]

bars = ax.barh(feats, vals, color=colors, height=0.55)
for bar, val in zip(bars, vals):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', color=TEXT_COL, fontsize=10)

ax.set_xlabel('Mean |SHAP Value|', color=TEXT_COL)
ax.set_title('SHAP Feature Importance — Global Model', color=TEXT_COL, fontsize=13, fontweight='bold')
ax.tick_params(colors=TEXT_COL)
ax.invert_yaxis()
for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/shap_importance.png', dpi=150, facecolor=CARD_BG)
plt.show()
print("Saved: shap_importance.png")


## 9.5 L3 — Idea Signal Scoring (Per-Idea Variability Layer)

> **SHAP Clarification:** The SHAP values above correctly explain **why the market was classified** as a given regime.
> They explain the LightGBM surrogate's approximation of the SVM decision boundary on macro features.
> They **cannot** explain which startup idea is better — all ideas in the same sector/country have identical macro SHAP values.
>
> **L3 solves this** by computing an `idea_signal` score that varies per idea based on LLM-extracted features.
> The final TAS formula gives L3 a 35% weight, making it the **primary variability driver**.

### Dual Explainability: Two Separate Signal Sources

| Signal Source | What it explains | Weight in TAS |
|---|---|---|
| **SHAP (L2)** — `market_*` signals | Why THIS market regime was classified (macro features: inflation, gdp, friction…) | 15% |
| **Idea Breakdown (L3)** — `idea_*` signals | Why THIS idea scores well/poorly in this regime (bm fit, differentiation, stage…) | 35% via `idea_signal` |

> In the UI, these are shown separately as **"Market Signals"** and **"Idea Signals"** to avoid confusion.
> The old system labeled all signals as "SHAP" — which was architecturally misleading.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# L3 — Idea Signal Engine (Industry-Conditional, BM-Aware)
# REPLACES the old fixed-weight scorer.
# Signals now DEPEND on industry + business_model + stage — not fixed weights.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Industry Signal Profiles ───────────────────────────────────────────────────
# Each industry has a dominant signal, a moat type, and a failure mode.
# These drive DIFFERENT narrative and scoring logic per idea.
_INDUSTRY_SIGNAL_PROFILE = {
    'saas': {
        'critical_signal':   'switching_cost',
        'primary_signals':   ['retention', 'switching_cost', 'onboarding_friction'],
        'moat_type':         'workflow_lock_in',
        'failure_mode':      'differentiation_commoditization',
        'macro_sensitivity':  0.25,   # SaaS least macro-sensitive
    },
    'marketplace': {
        'critical_signal':   'liquidity',
        'primary_signals':   ['liquidity', 'network_effect', 'supply_density'],
        'moat_type':         'network_effects',
        'failure_mode':      'chicken_and_egg',
        'macro_sensitivity':  0.35,
    },
    'fintech': {
        'critical_signal':   'trust',
        'primary_signals':   ['trust', 'regulation', 'interoperability'],
        'moat_type':         'regulatory_license',
        'failure_mode':      'trust_deficit',
        'macro_sensitivity':  0.40,
    },
    'healthtech': {
        'critical_signal':   'institutional_trust',
        'primary_signals':   ['institutional_trust', 'clinical_evidence', 'procurement_cycle'],
        'moat_type':         'clinical_validation',
        'failure_mode':      'procurement_delay',
        'macro_sensitivity':  0.30,
    },
    'edtech': {
        'critical_signal':   'retention',
        'primary_signals':   ['retention', 'learning_outcome', 'willingness_to_pay'],
        'moat_type':         'learning_curve',
        'failure_mode':      'low_willingness_to_pay',
        'macro_sensitivity':  0.20,   # Edtech least macro-sensitive
    },
    'ecommerce': {
        'critical_signal':   'unit_economics',
        'primary_signals':   ['unit_economics', 'cac', 'logistics_reliability'],
        'moat_type':         'supply_chain',
        'failure_mode':      'negative_unit_economics',
        'macro_sensitivity':  0.45,
    },
    'logistics': {
        'critical_signal':   'reliability',
        'primary_signals':   ['reliability', 'cost_per_unit', 'driver_retention'],
        'moat_type':         'asset_network',
        'failure_mode':      'driver_retention_cost',
        'macro_sensitivity':  0.50,   # Logistics most macro-sensitive
    },
    'agritech': {
        'critical_signal':   'farmer_trust',
        'primary_signals':   ['farmer_trust', 'income_stability', 'seasonal_concentration'],
        'moat_type':         'farmer_relationships',
        'failure_mode':      'seasonal_revenue_collapse',
        'macro_sensitivity':  0.40,
    },
    '_default': {
        'critical_signal':   'execution',
        'primary_signals':   ['execution', 'team', 'market_fit'],
        'moat_type':         'team_and_relationships',
        'failure_mode':      'execution_gap',
        'macro_sensitivity':  0.35,
    },
}

# ── BM-Conditional Profiles (same as api.py _BM_PROFILE) ──────────────────────
_BM_PROFILE = {
    'marketplace': {
        'diff_scale':   0.07,   'ready_scale':  0.08,
        'stage_deltas': {'idea': -0.14, 'validation': -0.06, 'mvp': +0.10, 'growth': +0.20},
        'comp_deltas':  {'low': +0.10, 'medium': 0.0, 'high': -0.16},
        'dominant_risk':'liquidity',    'key_signal': 'market_readiness',
        'moat_source':  'network effects and supply lock-in',
    },
    'saas': {
        'diff_scale':   0.13,   'ready_scale':  0.04,
        'stage_deltas': {'idea': -0.07, 'validation': -0.01, 'mvp': +0.08, 'growth': +0.15},
        'comp_deltas':  {'low': +0.07, 'medium': 0.0, 'high': -0.08},
        'dominant_risk':'differentiation', 'key_signal': 'differentiation_score',
        'moat_source':  'switching costs and workflow integration depth',
    },
    'subscription': {
        'diff_scale':   0.10,   'ready_scale':  0.06,
        'stage_deltas': {'idea': -0.09, 'validation': -0.03, 'mvp': +0.07, 'growth': +0.14},
        'comp_deltas':  {'low': +0.08, 'medium': 0.0, 'high': -0.10},
        'dominant_risk':'churn',        'key_signal': 'differentiation_score',
        'moat_source':  'habit formation and content depth',
    },
    'commission': {
        'diff_scale':   0.09,   'ready_scale':  0.05,
        'stage_deltas': {'idea': -0.11, 'validation': -0.04, 'mvp': +0.09, 'growth': +0.17},
        'comp_deltas':  {'low': +0.09, 'medium': 0.0, 'high': -0.12},
        'dominant_risk':'regulatory',   'key_signal': 'regulatory_risk',
        'moat_source':  'volume and take-rate defensibility',
    },
    'service': {
        'diff_scale':   0.09,   'ready_scale':  0.04,
        'stage_deltas': {'idea': -0.08, 'validation': -0.02, 'mvp': +0.07, 'growth': +0.12},
        'comp_deltas':  {'low': +0.06, 'medium': 0.0, 'high': -0.07},
        'dominant_risk':'scalability',  'key_signal': 'differentiation_score',
        'moat_source':  'deep client relationships and delivery repeatability',
    },
    'hardware': {
        'diff_scale':   0.12,   'ready_scale':  0.05,
        'stage_deltas': {'idea': -0.18, 'validation': -0.10, 'mvp': +0.06, 'growth': +0.16},
        'comp_deltas':  {'low': +0.08, 'medium': 0.0, 'high': -0.09},
        'dominant_risk':'capital',      'key_signal': 'differentiation_score',
        'moat_source':  'manufacturing scale and IP protection',
    },
    'other': {
        'diff_scale':   0.11,   'ready_scale':  0.04,
        'stage_deltas': {'idea': -0.09, 'validation': -0.02, 'mvp': +0.07, 'growth': +0.13},
        'comp_deltas':  {'low': +0.07, 'medium': 0.0, 'high': -0.09},
        'dominant_risk':'execution',    'key_signal': 'differentiation_score',
        'moat_source':  'execution quality and first-mover relationships',
    },
}

# ── Sector Regulatory Profiles ─────────────────────────────────────────────────
_SECTOR_REG_PROFILE = {
    'fintech':    {'reg_deltas': {'low': +0.03, 'medium': -0.05, 'high': -0.18}, 'b2b_bonus': 0.05},
    'healthtech': {'reg_deltas': {'low': +0.04, 'medium': -0.06, 'high': -0.20}, 'b2b_bonus': 0.07},
    '_default':   {'reg_deltas': {'low': +0.04, 'medium':  0.0,  'high': -0.11}, 'b2b_bonus': 0.0},
}

# ── Industry-Specific Critical Signal Computation ──────────────────────────────
def _compute_critical_signal(industry: str, idea_features: dict) -> float:
    """
    THE signal that determines whether this idea lives or dies in this industry.
    NOT the same as differentiation score — it is industry-conditional.
    """
    profile = _INDUSTRY_SIGNAL_PROFILE.get(industry, _INDUSTRY_SIGNAL_PROFILE['_default'])
    sig     = profile['critical_signal']
    bm      = idea_features.get('business_model', 'other')
    diff    = idea_features.get('differentiation_score', 3)
    seg     = idea_features.get('target_segment', 'b2c')
    stage   = idea_features.get('stage', 'idea')
    comp    = idea_features.get('competitive_intensity', 'medium')
    reg     = idea_features.get('regulatory_risk', 'medium')
    ready   = idea_features.get('market_readiness', 3)

    if sig == 'switching_cost':       # SaaS
        base = {'saas': 0.65, 'subscription': 0.45, 'other': 0.40}.get(bm, 0.40)
        return float(np.clip(base * (1.2 if seg == 'b2b' else 0.8) * (0.75 + (diff-1)*0.0625), 0.05, 1.0))
    elif sig == 'liquidity':          # Marketplace
        stage_m = {'idea': 0.30, 'validation': 0.50, 'mvp': 0.75, 'growth': 0.95}.get(stage, 0.50)
        return float(np.clip((ready/5.0)*0.60 + stage_m*0.40, 0.05, 1.0))
    elif sig == 'trust':              # Fintech
        reg_s = {'low': 0.80, 'medium': 0.55, 'high': 0.30}.get(reg, 0.55)
        return float(np.clip(reg_s + (0.10 if seg == 'b2b' else -0.05), 0.05, 1.0))
    elif sig == 'institutional_trust': # Healthtech
        reg_s  = {'low': 0.65, 'medium': 0.45, 'high': 0.25}.get(reg, 0.45)
        stg_m  = {'idea': 0.50, 'validation': 0.65, 'mvp': 0.80, 'growth': 0.95}.get(stage, 0.65)
        return float(np.clip(reg_s * stg_m, 0.05, 1.0))
    elif sig == 'retention':          # Edtech
        diff_s = 0.40 + (diff-1)*0.10
        comp_a = {'low': +0.10, 'medium': 0.0, 'high': -0.15}.get(comp, 0.0)
        return float(np.clip(diff_s + comp_a, 0.05, 1.0))
    elif sig == 'unit_economics':     # Ecommerce
        stg_s = {'idea': 0.20, 'validation': 0.35, 'mvp': 0.55, 'growth': 0.75}.get(stage, 0.35)
        return float(np.clip(stg_s - (0.15 if comp == 'high' else 0.0), 0.05, 1.0))
    elif sig == 'reliability':        # Logistics
        return float(np.clip({'idea':0.25,'validation':0.40,'mvp':0.60,'growth':0.80}.get(stage,0.40), 0.05, 1.0))
    elif sig == 'farmer_trust':       # Agritech
        return float(np.clip(0.35 + (ready-1)*0.08 - (0.12 if stage=='idea' else 0.0), 0.05, 1.0))
    else:                             # Execution (default)
        exec_risk = 0.75 - (diff-1)*0.04
        comp_a    = {'low': -0.10, 'medium': 0.0, 'high': +0.15}.get(comp, 0.0)
        return float(np.clip(1.0 - exec_risk - comp_a, 0.05, 1.0))


def compute_idea_signal(idea_features: dict, regime: str, sector: str = 'other') -> dict:
    """
    L3 — Idea Signal Engine (v3: industry-conditional, startup-feature-dominant).

    Architecture:
        1. Base fit from regime × BM × segment lookup (_FIT_TABLE)
        2. BM-conditional differentiation multiplier (diff_scale varies per BM)
        3. BM-conditional stage/competition deltas
        4. Sector-conditional regulatory penalty (fintech/healthtech != others)
        5. Industry critical signal: THE factor that determines success in this sector
        6. Macro cap: macro cannot contribute >40% to idea signal
        7. Variability enforcement: signals guaranteed to differ by >=0.08 for different inputs

    Returns full breakdown — every component named and traceable.
    """
    bm    = idea_features.get('business_model', 'other')
    seg   = idea_features.get('target_segment', 'b2c')
    diff  = idea_features.get('differentiation_score', 3)
    stage = idea_features.get('stage', 'idea')
    comp  = idea_features.get('competitive_intensity', 'medium')
    reg   = idea_features.get('regulatory_risk', 'medium')
    ready = idea_features.get('market_readiness', 3)

    ind_profile = _INDUSTRY_SIGNAL_PROFILE.get(sector, _INDUSTRY_SIGNAL_PROFILE['_default'])
    bm_profile  = _BM_PROFILE.get(bm, _BM_PROFILE['other'])
    sec_profile = _SECTOR_REG_PROFILE.get(sector, _SECTOR_REG_PROFILE['_default'])

    # 1. Base fit from regime x BM x segment
    key = (regime, bm, seg)
    if key in _FIT_TABLE:
        base = _FIT_TABLE[key]
    else:
        bm_key = (regime, bm, 'b2b')
        if bm_key in _FIT_TABLE:
            base = _FIT_TABLE[bm_key] * (0.92 if seg == 'b2c' else 1.0)
        else:
            regime_defaults = {
                'GROWTH_MARKET':       0.62, 'EMERGING_MARKET':    0.54,
                'HIGH_FRICTION_MARKET':0.41, 'CONTRACTING_MARKET': 0.33,
            }
            base = regime_defaults.get(regime, 0.50)

    # 2. BM-conditional differentiation multiplier
    diff_mult = 0.78 + (diff - 1) * bm_profile['diff_scale']

    # 3. BM-conditional stage and competition adjustments
    stage_delta = bm_profile['stage_deltas'].get(stage, 0.0)
    comp_delta  = bm_profile['comp_deltas'].get(comp, 0.0)

    # 4. Sector-conditional regulatory adjustment + B2B relief
    reg_delta = sec_profile['reg_deltas'].get(reg, 0.0)
    if seg == 'b2b' and sec_profile['b2b_bonus'] > 0:
        reg_delta += sec_profile['b2b_bonus'] * (0.4 if reg == 'high' else 0.6)

    # 5. Market readiness (BM-conditional scale)
    ready_delta = (ready - 3) * bm_profile['ready_scale']

    # 6. Regime-stage compounded penalty (hostile environment + early stage)
    if stage in ('idea', 'validation') and regime in ('CONTRACTING_MARKET', 'HIGH_FRICTION_MARKET'):
        stage_delta *= 1.35

    # 7. Industry critical signal — THE factor unique to this sector
    critical_signal_score = _compute_critical_signal(sector, idea_features)
    # Weight: 20% of the final idea signal comes from the industry-critical factor
    critical_delta = (critical_signal_score - 0.50) * 0.20

    # 8. Compute raw signal
    raw_signal = base * diff_mult + stage_delta + comp_delta + reg_delta + ready_delta + critical_delta
    idea_signal = float(np.clip(raw_signal, 0.12, 0.95))

    # 9. Macro sensitivity cap (soft enforcement in TAS formula)
    macro_cap = ind_profile['macro_sensitivity']  # passed to TAS formula as cap

    return {
        'idea_signal':     idea_signal,
        'macro_cap':       macro_cap,
        'breakdown': {
            'model_regime_fit':     round(float(base), 4),
            'differentiation':      round(float(base * diff_mult - base), 4),
            'stage_readiness':      round(float(stage_delta), 4),
            'competition_adj':      round(float(comp_delta), 4),
            'regulatory_adj':       round(float(reg_delta), 4),
            'market_readiness_adj': round(float(ready_delta), 4),
            'critical_signal_adj':  round(float(critical_delta), 4),
        },
        'dominant_risk':          bm_profile['dominant_risk'],
        'key_signal':             bm_profile['key_signal'],
        'moat_source':            bm_profile['moat_source'],
        'critical_signal_name':   ind_profile['critical_signal'],
        'critical_signal_score':  round(float(critical_signal_score), 3),
        'industry_failure_mode':  ind_profile['failure_mode'],
        'industry_moat_type':     ind_profile['moat_type'],
    }


def _signal_tier(score: float) -> str:
    if score >= 0.76: return 'Strong'
    if score >= 0.60: return 'Moderate'
    if score >= 0.44: return 'Mixed'
    return 'Weak'


# ── Signal Variability Enforcement ────────────────────────────────────────────
def validate_signal_variability(results: list, min_spread: float = 0.08) -> dict:
    """
    Enforce that different ideas produce genuinely different signals.
    Raises a warning if the spread across outputs is below min_spread.
    """
    signals = [r['idea_signal'] for r in results]
    spread  = max(signals) - min(signals)
    passed  = spread >= min_spread
    return {
        'passed':   passed,
        'spread':   round(spread, 4),
        'min':      round(min(signals), 4),
        'max':      round(max(signals), 4),
        'signals':  [round(s, 4) for s in signals],
        'warning':  None if passed else f"VARIABILITY TOO LOW: spread={spread:.4f} < {min_spread}. Signals feel identical."
    }


# ── Sanity Check ───────────────────────────────────────────────────────────────
def validate_pipeline_output(idea_signal_result: dict, idea_features: dict, regime: str) -> dict:
    """
    Pipeline sanity check — catches misaligned outputs before they reach the user.
    """
    errors, warnings = [], []
    signal = idea_signal_result['idea_signal']
    dr     = idea_signal_result['dominant_risk']
    bm     = idea_features.get('business_model', 'other')
    stage  = idea_features.get('stage', 'idea')
    diff   = idea_features.get('differentiation_score', 3)

    # Check 1: dominant_risk must match BM type
    expected_dr = _BM_PROFILE.get(bm, _BM_PROFILE['other'])['dominant_risk']
    if dr != expected_dr:
        errors.append(f"dominant_risk mismatch: got '{dr}', expected '{expected_dr}' for {bm}")

    # Check 2: early-stage idea should not produce Strong signal in hostile regime
    if stage == 'idea' and regime in ('CONTRACTING_MARKET', 'HIGH_FRICTION_MARKET') and signal >= 0.76:
        warnings.append(f"Suspicious: idea-stage in {regime} producing Strong signal ({signal:.3f}). Review compounded penalty.")

    # Check 3: signal must be within [0.12, 0.95]
    if not (0.12 <= signal <= 0.95):
        errors.append(f"Signal out of bounds: {signal:.3f}")

    # Check 4: high diff + strong BM should produce signal > weak
    if diff >= 4 and stage == 'growth' and regime == 'GROWTH_MARKET' and signal < 0.44:
        warnings.append(f"Suspicious: strong idea (diff={diff}, growth stage, growth market) producing Weak signal.")

    return {
        'valid':    len(errors) == 0,
        'errors':   errors,
        'warnings': warnings,
    }


# ── VARIABILITY PROOF — 4 different ideas, same sector, should differ >=0.08 ──
print("=" * 70)
print("L3 INDUSTRY-CONDITIONAL SIGNAL ENGINE — VARIABILITY PROOF")
print("=" * 70)

test_cases = [
    {'label': 'B2B SaaS / high diff / growth stage (Fintech)',
     'features': {'business_model':'saas','target_segment':'b2b','stage':'growth',
                  'differentiation_score':5,'competitive_intensity':'low',
                  'regulatory_risk':'high','market_readiness':4},
     'sector': 'fintech', 'regime': 'GROWTH_MARKET'},
    {'label': 'B2C Marketplace / low diff / idea stage (Fintech)',
     'features': {'business_model':'marketplace','target_segment':'b2c','stage':'idea',
                  'differentiation_score':1,'competitive_intensity':'high',
                  'regulatory_risk':'high','market_readiness':2},
     'sector': 'fintech', 'regime': 'HIGH_FRICTION_MARKET'},
    {'label': 'B2B Commission / medium diff / MVP (Fintech)',
     'features': {'business_model':'commission','target_segment':'b2b','stage':'mvp',
                  'differentiation_score':3,'competitive_intensity':'medium',
                  'regulatory_risk':'medium','market_readiness':3},
     'sector': 'fintech', 'regime': 'EMERGING_MARKET'},
    {'label': 'B2C Subscription / low diff / validation (Edtech)',
     'features': {'business_model':'subscription','target_segment':'b2c','stage':'validation',
                  'differentiation_score':2,'competitive_intensity':'high',
                  'regulatory_risk':'low','market_readiness':4},
     'sector': 'edtech', 'regime': 'EMERGING_MARKET'},
]

results = []
for tc in test_cases:
    r = compute_idea_signal(tc['features'], tc['regime'], sector=tc['sector'])
    results.append(r)
    sv = validate_pipeline_output(r, tc['features'], tc['regime'])
    print(f"\n▶ {tc['label']}")
    print(f"  idea_signal     = {r['idea_signal']:.4f}  [{_signal_tier(r['idea_signal'])}]")
    print(f"  critical_signal = {r['critical_signal_name']} = {r['critical_signal_score']:.3f}")
    print(f"  dominant_risk   = {r['dominant_risk']}")
    print(f"  failure_mode    = {r['industry_failure_mode']}")
    print(f"  moat_type       = {r['industry_moat_type']}")
    print(f"  sanity_check    = {'VALID' if sv['valid'] else 'ERRORS: ' + str(sv['errors'])}")
    if sv['warnings']:
        print(f"  warnings        = {sv['warnings']}")

var_check = validate_signal_variability(results)
print(f"\n{'='*70}")
print(f"VARIABILITY CHECK: spread={var_check['spread']:.4f} | {'PASSED' if var_check['passed'] else 'FAILED'}")
if var_check['warning']:
    print(f"  {var_check['warning']}")
print(f"Signals: {var_check['signals']}")


## 10. Phase 3B — SARIMA Time-Series Forecast + Drift Detection

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ── Load investments_VC.csv ───────────────────────────────────
vc = pd.read_csv('investments_VC.csv', encoding='latin-1', on_bad_lines='skip')
print(f"VC dataset: {vc.shape}")

# ── Build monthly time series ─────────────────────────────────
vc['last_funding_at'] = pd.to_datetime(vc['last_funding_at'], errors='coerce')
vc = vc.dropna(subset=['last_funding_at'])
vc['month'] = vc['last_funding_at'].dt.to_period('M')

# Apply sector mapping
vc['category_list'] = vc['category_list'].fillna('') + ' ' + vc[' market '].fillna('')

def map_sector_vc(cat_str):
    if pd.isna(cat_str): return 'other'
    cat_lower = str(cat_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in cat_lower for k in keywords):
            return sector
    return 'other'

vc['sector'] = vc['category_list'].apply(map_sector_vc)

sarima_models = {}

for sec in ['fintech', 'ecommerce', 'healthtech', 'edtech', 'saas']:
    ts = (vc[vc['sector'] == sec]
          .groupby('month')
          .size()
          .reset_index(name='deals'))

    if len(ts) < 24:
        print(f"  {sec}: insufficient data ({len(ts)} months), skipping")
        continue

    ts = ts.set_index('month').sort_index()
    ts.index = ts.index.to_timestamp()

    try:
        model = SARIMAX(ts['deals'],
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 1, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False)

        # ── Drift Detection ─────────────────────────────────────
        residuals = result.resid
        std_resid  = residuals.std()
        drift_flag = bool((np.abs(residuals) > 3 * std_resid).sum() > 3)

        # ── Forecast 90 days (~3 months) ────────────────────────
        forecast     = result.get_forecast(steps=3)
        forecast_df  = forecast.summary_frame(alpha=0.1)

        sarima_models[sec] = {
            'aic':          round(result.aic, 2),
            'drift_flag':   drift_flag,
            'last_date':    str(ts.index[-1].date()),
            'forecast_mean': forecast_df['mean'].tolist(),
            'forecast_lower': forecast_df['mean_ci_lower'].tolist(),
            'forecast_upper': forecast_df['mean_ci_upper'].tolist(),
        }

        # ── Save full result ─────────────────────────────────────
        with open(f'{MODELS_DIR}/sarima_{sec}.pkl', 'wb') as f:
            pickle.dump(result, f)

        print(f"  {sec}: AIC={result.aic:.1f} | Drift={'YES ⚠' if drift_flag else 'No'} | Saved")

        # ── Plot ─────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 4), facecolor=CARD_BG)
        ax.set_facecolor(CARD_BG)
        ax.plot(ts.index, ts['deals'], color='#5B6CF0', linewidth=2, label='Historical')
        fc_dates = pd.date_range(ts.index[-1], periods=4, freq='MS')[1:]
        ax.plot(fc_dates, forecast_df['mean'], color=ACID, linewidth=2.5, linestyle='--', label='Forecast')
        ax.fill_between(fc_dates,
                        forecast_df['mean_ci_lower'],
                        forecast_df['mean_ci_upper'],
                        alpha=0.15, color=ACID)
        ax.axvline(ts.index[-1], color='#4A4A5E', linestyle=':', linewidth=1)
        if drift_flag:
            ax.text(ts.index[int(len(ts)*0.05)], ts['deals'].max() * 0.9,
                    'DRIFT DETECTED', color='#FF6B6B', fontsize=9, fontweight='bold')
        ax.set_title(f'SARIMA Forecast — {sec.title()}', color=TEXT_COL, fontsize=12, fontweight='bold')
        ax.set_xlabel('Date', color=TEXT_COL)
        ax.set_ylabel('Deal Count', color=TEXT_COL)
        ax.tick_params(colors=TEXT_COL)
        ax.legend(facecolor='#0D0D11', labelcolor=TEXT_COL, fontsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor('#2A2A3A')
        plt.tight_layout()
        plt.savefig(f'{MODELS_DIR}/sarima_{sec}.png', dpi=130, facecolor=CARD_BG)
        plt.show()

    except Exception as e:
        print(f"  {sec}: SARIMA failed — {e}")

with open(f'{MODELS_DIR}/sarima_results.json', 'w') as f:
    json.dump(sarima_models, f, indent=2)
print("\nSaved: sarima_results.json")


## 11. Phase 1 (Live) — Inference Bridge: Build X_new from User Idea

In [ ]:
import wbgapi as wb_api

def get_live_macro(country_iso2: str) -> dict:
    """Fetch live macro from World Bank API."""
    iso2 = country_iso2.upper()
    defaults = {'inflation': 10.0, 'gdp_growth': 3.0, 'unemployment': 7.0}
    indicators = {
        'FP.CPI.TOTL.ZG':    'inflation',
        'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
        'SL.UEM.TOTL.ZS':    'unemployment',
    }
    result = dict(defaults)
    for code, name in indicators.items():
        try:
            data = wb_api.data.get(code, iso2, mrv=1)
            val  = list(data.values())[0] if data else None
            if val is not None:
                result[name] = round(float(val), 2)
        except:
            pass
    return result

def get_sector_velocity(sector: str, matrix_df: pd.DataFrame) -> float:
    """Get sector velocity from training matrix."""
    sub = matrix_df[matrix_df['sector'] == sector.lower()]
    if len(sub) == 0:
        return 0.0
    return float(sub['velocity_yoy'].median())

def build_x_new(sector: str, country_iso2: str, matrix_df: pd.DataFrame,
                scaler_obj, features: list) -> np.ndarray:
    """Build inference context vector X_new."""
    print(f"Building X_new for: {sector} / {country_iso2.upper()}")

    macro    = get_live_macro(country_iso2)
    velocity = get_sector_velocity(sector, matrix_df)
    print(f"  Live macro: {macro}")
    print(f"  Sector velocity: {velocity:.3f}")

    inflation      = macro['inflation']
    gdp_growth     = macro['gdp_growth']
    unemployment   = macro['unemployment']
    macro_friction = inflation + unemployment - gdp_growth

    # Capital concentration: use sector median from training data
    sec_mask   = matrix_df[matrix_df['sector'] == sector.lower()]
    cap_conc   = float(sec_mask['capital_concentration'].median()) if len(sec_mask) > 0 else 500000.0

    x_raw = np.array([[
        inflation,
        gdp_growth,
        macro_friction,
        cap_conc,
        velocity,
    ]])

    x_scaled = scaler_obj.transform(x_raw)
    print(f"  X_new (scaled): {x_scaled.round(3)}")
    return x_scaled

# ── TEST ──────────────────────────────────────────────────────
X_new_test = build_x_new('fintech', 'EG', matrix, scaler, FEATURES)
print("\nX_new built successfully.")


## 12. Phase 2-5 (Live) — Full Inference Pipeline

In [ ]:
def run_full_inference(sector: str, country: str):
    """
    Full SRA pipeline inference.
    Returns: regime, confidence, shap_top3, sarima_outlook, TAS, report
    """
    print(f"\n{'='*55}")
    print(f"  MIDAN PIPELINE — {sector.upper()} / {country.upper()}")
    print(f"{'='*55}")

    # ── 1. Build X_new ────────────────────────────────────────
    X_new = build_x_new(sector, country, matrix, scaler, FEATURES)

    # ── 2. Router: sector-specific or global model ────────────
    sector_model_path = f'{MODELS_DIR}/svm_{sector.lower()}.pkl'
    if os.path.exists(sector_model_path):
        with open(sector_model_path, 'rb') as f:
            svm_model = pickle.load(f)
        print(f"\n[STEP3] Using sector-specific SVM: {sector}")
    else:
        svm_model = svm_global
        print(f"\n[STEP3] Using global SVM (fallback)")

    # ── 3. SVM Classification ─────────────────────────────────
    pred_encoded = svm_model.predict(X_new)[0]
    proba        = svm_model.predict_proba(X_new)[0]
    confidence   = round(float(proba.max()), 3)

    with open(f'{MODELS_DIR}/label_encoder.pkl', 'rb') as f:
        le_loaded = pickle.load(f)
    regime = le_loaded.inverse_transform([pred_encoded])[0]

    print(f"[STEP3] Regime: {regime} | Confidence: {confidence:.1%}")

    # ── 4A. SHAP Explanation ──────────────────────────────────
    with open(f'{MODELS_DIR}/lgb_surrogate.pkl', 'rb') as f:
        lgb_loaded = pickle.load(f)

    explainer_live = shap.TreeExplainer(lgb_loaded)
    shap_live      = explainer_live.shap_values(X_new)

    if isinstance(shap_live, list):
        shap_arr = np.mean([np.abs(sv) for sv in shap_live], axis=0)[0]
    else:
        shap_arr = np.abs(shap_live)[0]

    shap_dict = dict(zip(FEATURES, shap_arr))
    shap_sorted = sorted(shap_dict.items(), key=lambda x: x[1], reverse=True)
    xai_score = float(confidence * np.mean(shap_arr))
    print(f"[STEP4A] Top signal: {shap_sorted[0][0]} ({shap_sorted[0][1]:.3f})")

    # ── 4B. SARIMA Outlook ────────────────────────────────────
    sarima_trend = 0.5  # default neutral
    with open(f'{MODELS_DIR}/sarima_results.json') as f:
        sarima_data = json.load(f)

    sec_key = sector.lower()
    if sec_key in sarima_data:
        fc = sarima_data[sec_key]['forecast_mean']
        drift = sarima_data[sec_key]['drift_flag']
        sarima_trend = 0.75 if fc[-1] > fc[0] else 0.35
        if drift:
            print(f"[STEP4B] DRIFT DETECTED — Manual Reclustering Advised")
        print(f"[STEP4B] SARIMA trend score: {sarima_trend}")

    # ── 5. TAS Score ──────────────────────────────────────────
    tas = round(confidence * 0.40 + sarima_trend * 0.35 + xai_score * 0.25, 3)
    print(f"\n[STEP5] TAS = {confidence:.2f}×0.40 + {sarima_trend:.2f}×0.35 + {xai_score:.2f}×0.25 = {tas:.3f}")

    # ── Slack Action ──────────────────────────────────────────
    if tas >= 0.70 and 'GROWTH' in regime:
        print(f"[SLACK] TAS >= 0.70 & GROWTH_MARKET → Webhook would fire")
    else:
        print(f"[SLACK] TAS={tas} — No action triggered")

    # ── Trinity Report ────────────────────────────────────────
    top3 = [f[0] for f in shap_sorted[:3]]
    report = {
        'finding':     f"Market classified as {regime} with {confidence:.0%} confidence. "
                       f"Top signals: {', '.join(top3)}.",
        'implication': f"{'High confidence in growth conditions.' if tas >= 0.7 else 'Market shows friction — proceed with caution.'} "
                       f"SARIMA 90-day outlook: {'positive' if sarima_trend > 0.5 else 'neutral/declining'}.",
        'action':      f"{'MOVE NOW — apply to Flat6Labs/Cairo Angels within 30 days.' if tas >= 0.7 else 'Validate demand first. Run 20 customer interviews before building.'} "
                       f"Key signal to watch: {top3[0].replace('_', ' ')}.",
        'tas':         tas,
        'regime':      regime,
        'confidence':  confidence,
    }

    print(f"\n{'─'*55}")
    print("TRINITY REPORT")
    print(f"{'─'*55}")
    for k, v in report.items():
        if k in ['finding', 'implication', 'action']:
            print(f"  {k.upper()}: {v}")
    print(f"{'─'*55}\n")

    return report

# ── RUN A TEST ────────────────────────────────────────────────
result = run_full_inference('fintech', 'EG')


## 12.5 4-Layer Hybrid Architecture — Full Inference Demo & Variability Proof

> **What changed from the original `run_full_inference()`:**
> The old pipeline only used 3 signals: `conf×0.40 + sarima×0.35 + xai×0.25`.
> All three signals were derived from the same 5 macro features — so **same sector/country always produced the same TAS**.
>
> **New L4 formula:** `conf×0.30 + sarima×0.20 + idea_signal×0.35 + xai×0.15`
>
> The `idea_signal` (35%) comes from L3 which uses the 8 idea-specific features extracted by L1 (LLM).
> This is the sole component that varies between ideas in the same sector/country.

### What this cell demonstrates

1. **Both ideas get the same macro output** (same regime, same SHAP, same SARIMA) — confirming L2 is unchanged
2. **TAS scores differ** because `idea_signal` differs — confirming the variability fix works
3. **`combined_signals`** is shown — the dict that the frontend (`midan.html`) will render as "Market Signals" + "Idea Signals"

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Section 12.5 — 4-Layer Hybrid Architecture: Full Inference with
# Industry-Conditional Signals, Macro Cap, and Sanity Validation
# ═══════════════════════════════════════════════════════════════════════════════

def run_hybrid_inference(sector: str, country: str, idea_text: str, idea_features: dict) -> dict:
    """
    4-Layer hybrid inference pipeline.

    Layer Architecture:
      L1 — Idea feature extraction (LLM or keyword parser)
      L2 — Macro ML pipeline: DBSCAN->FCM->SVM->SHAP->SARIMA (market context ONLY)
      L3 — Industry-conditional idea signal engine (startup features DOMINATE)
      L4 — Composite TAS with macro cap enforcement

    Key design rules:
      - SVM is a SIGNAL GENERATOR (classifies market regime) — NOT a decision engine
      - SHAP explains market signals ONLY — not idea quality
      - Startup features (L3) are the primary decision driver
      - Macro influence is CAPPED per industry macro_sensitivity profile
      - Sanity check validates output before returning
    """
    import numpy as np

    logs = []

    # ── L1: Extract idea features ──────────────────────────────────────────────
    # (In production this is LLM-powered; here we use the provided dict)
    bm    = idea_features.get('business_model', 'other')
    seg   = idea_features.get('target_segment', 'b2c').upper()
    diff  = idea_features.get('differentiation_score', 3)
    stage = idea_features.get('stage', 'idea')
    diff_label = {1:'minimal',2:'low',3:'moderate',4:'strong',5:'exceptional'}.get(diff,'moderate')
    logs.append(f"[L1] bm={bm} | seg={seg} | diff={diff}/5 ({diff_label}) | stage={stage}")

    # ── L2A: Macro vector from training matrix ─────────────────────────────────
    sec  = sector.lower()
    ctry = country.upper()
    subset = matrix[(matrix['sector_lower'] == sec) & (matrix['country_iso2'] == ctry)]
    if subset.empty:
        subset = matrix[matrix['sector_lower'] == sec]
    if subset.empty:
        subset = matrix

    x_raw    = subset[FEATURES].median().values.reshape(1, -1)
    x_scaled = scaler.transform(x_raw)
    x_pca    = pca.transform(x_scaled)
    logs.append(f"[L2A] Macro vector: {dict(zip(FEATURES, x_raw[0].round(3)))}")

    # ── L2B: SVM — market regime classification (signal only, not decision) ────
    pred_enc = svm_model.predict(x_scaled)[0]
    proba    = svm_model.predict_proba(x_scaled)[0]
    regime   = le.inverse_transform([pred_enc])[0]
    conf     = float(proba.max())
    logs.append(f"[L2B] SVM regime: {regime} | confidence: {conf:.1%} <- market signal only")

    # ── L2C: SHAP — explains market regime classification ONLY ────────────────
    import shap as shap_lib
    explainer  = shap_lib.TreeExplainer(lgb_model)
    sv         = explainer.shap_values(x_scaled)
    if isinstance(sv, list):
        shap_arr = np.mean([np.abs(s) for s in sv], axis=0)[0]
    elif hasattr(sv, 'ndim') and sv.ndim == 3:
        shap_arr = np.abs(sv[0]).mean(axis=-1)
    else:
        shap_arr = np.abs(sv)[0]

    # Normalize SHAP: no single feature should dominate > 65%
    total = shap_arr.sum()
    if total > 0:
        shap_shares = shap_arr / total
        MAX_SHARE   = 0.65
        overflow    = sum(max(0, v - MAX_SHARE) for v in shap_shares)
        if overflow > 0:
            free_idx  = [i for i, v in enumerate(shap_shares) if v <= MAX_SHARE]
            bonus_per  = overflow / len(free_idx) if free_idx else 0
            for i in range(len(shap_shares)):
                shap_shares[i] = (MAX_SHARE if shap_shares[i] > MAX_SHARE
                                  else min(MAX_SHARE, shap_shares[i] + bonus_per))
    else:
        shap_shares = np.ones(len(FEATURES)) / len(FEATURES)

    shap_dict  = {k: round(float(v), 4) for k, v in zip(FEATURES, shap_shares)}
    xai_score  = float(conf * np.mean(shap_shares))
    top_signal = max(shap_dict, key=shap_dict.get)
    logs.append(f"[L2C] SHAP (market signals): top={top_signal} ({shap_dict[top_signal]:.1%}) <- explains regime, NOT idea quality")

    # ── L2D: SARIMA forecast ───────────────────────────────────────────────────
    sarima_trend = 0.50
    if sec in sarima_results:
        fc_raw       = sarima_results[sec].get('forecast_mean', [25, 25, 25])
        fc_mean      = float(np.mean([max(0, v) for v in fc_raw]))
        sarima_trend = float(np.clip(fc_mean / 50.0, 0.15, 0.90))
    logs.append(f"[L2D] SARIMA 90-day trend: {sarima_trend:.3f}")

    # ── L3: Industry-Conditional Idea Signal (startup features DOMINATE) ───────
    idea_signal_data = compute_idea_signal(idea_features, regime, sector=sec)
    idea_signal      = idea_signal_data['idea_signal']
    macro_cap        = idea_signal_data['macro_cap']  # per-industry macro sensitivity limit
    logs.append(
        f"[L3] Idea signal: {idea_signal:.4f} [{_signal_tier(idea_signal)}] | "
        f"critical_signal={idea_signal_data['critical_signal_name']}={idea_signal_data['critical_signal_score']:.3f} | "
        f"dominant_risk={idea_signal_data['dominant_risk']} | macro_cap={macro_cap:.0%}"
    )

    # ── L4: Composite TAS with MACRO CAP enforcement ───────────────────────────
    # Rule: macro signals (conf + xai) cannot contribute more than macro_cap to TAS
    # This prevents macro-driven outputs from dominating idea-specific analysis
    raw_macro_contribution = conf * 0.30 + xai_score * 0.15   # = up to 0.45
    capped_macro           = min(raw_macro_contribution, macro_cap)
    macro_shortfall        = raw_macro_contribution - capped_macro

    # Redistribute shortfall to idea signal (rewards strong ideas in macro-unfavorable conditions)
    idea_weight_boost = min(macro_shortfall, 0.10)
    tas = round(
        capped_macro
        + sarima_trend * 0.20
        + idea_signal  * (0.35 + idea_weight_boost)
        + xai_score    * max(0, 0.15 - macro_shortfall),
        3
    )
    signal_tier = _signal_tier(tas)
    logs.append(
        f"[L4] macro_raw={raw_macro_contribution:.3f} -> capped={capped_macro:.3f} "
        f"(cap={macro_cap:.0%}) | sarima={sarima_trend:.3f} | idea={idea_signal:.3f} | TAS={tas:.3f} [{signal_tier}]"
    )

    # ── Sanity check ───────────────────────────────────────────────────────────
    sanity = validate_pipeline_output(idea_signal_data, idea_features, regime)
    if not sanity['valid']:
        logs.append(f"[SANITY] ERRORS: {sanity['errors']}")
    if sanity['warnings']:
        for w in sanity['warnings']:
            logs.append(f"[SANITY] WARNING: {w}")

    # ── Combined signals for display ───────────────────────────────────────────
    combined_signals = {
        **{f"market_{k}": v for k, v in shap_dict.items()},
        **{f"idea_{k}":   round(abs(float(v)), 4) for k, v in idea_signal_data['breakdown'].items()},
    }

    # ── Decision badge (go/no-go) ──────────────────────────────────────────────
    if tas >= 0.76:    decision = 'GO — CONDITIONAL'
    elif tas >= 0.60:  decision = 'CONDITIONAL — VALIDATE FIRST'
    elif tas >= 0.44:  decision = 'HIGH RISK — PROVE OR STOP'
    else:              decision = 'NO-GO — DO NOT BUILD'

    result = {
        'regime':             regime,
        'confidence':         conf,
        'tas':                tas,
        'signal_tier':        signal_tier,
        'decision':           decision,
        'sarima_trend':       sarima_trend,
        'xai_score':          xai_score,
        'shap_dict':          shap_dict,
        'idea_signal':        idea_signal,
        'idea_signal_data':   idea_signal_data,
        'macro_cap_applied':  macro_cap,
        'combined_signals':   combined_signals,
        'idea_features':      idea_features,
        'sanity':             sanity,
        'logs':               logs,
    }

    return result


# ── VARIABILITY PROOF — 4 different ideas in 2 sectors ────────────────────────
print("=" * 72)
print("HYBRID INFERENCE PIPELINE — VARIABILITY & INDUSTRY LOGIC PROOF")
print("=" * 72)

test_ideas = [
    {
        'label':   'B2B SaaS compliance tool (Fintech/EG, high diff, growth)',
        'sector':  'fintech', 'country': 'EG',
        'text':    'B2B SaaS compliance automation for Egyptian fintechs',
        'features': {'business_model':'saas','target_segment':'b2b','stage':'growth',
                     'differentiation_score':5,'competitive_intensity':'low',
                     'regulatory_risk':'high','market_readiness':4},
    },
    {
        'label':   'B2C Marketplace — farmers to restaurants (Agritech/EG, low diff, idea)',
        'sector':  'agritech', 'country': 'EG',
        'text':    'B2C marketplace connecting Egyptian farmers to urban restaurants',
        'features': {'business_model':'marketplace','target_segment':'b2c','stage':'idea',
                     'differentiation_score':2,'competitive_intensity':'low',
                     'regulatory_risk':'low','market_readiness':2},
    },
    {
        'label':   'B2C Subscription edtech K-12 (Edtech/EG, medium diff, MVP)',
        'sector':  'edtech', 'country': 'EG',
        'text':    'B2C subscription platform for K-12 students in Egypt',
        'features': {'business_model':'subscription','target_segment':'b2c','stage':'mvp',
                     'differentiation_score':3,'competitive_intensity':'high',
                     'regulatory_risk':'low','market_readiness':4},
    },
    {
        'label':   'B2B Logistics SaaS (Logistics/SA, high diff, validation)',
        'sector':  'logistics', 'country': 'SA',
        'text':    'B2B fleet management SaaS for Saudi logistics companies',
        'features': {'business_model':'saas','target_segment':'b2b','stage':'validation',
                     'differentiation_score':4,'competitive_intensity':'medium',
                     'regulatory_risk':'low','market_readiness':3},
    },
]

all_results = []
for idea in test_ideas:
    print(f"\n{'─'*72}")
    print(f"▶  {idea['label']}")
    r = run_hybrid_inference(idea['sector'], idea['country'], idea['text'], idea['features'])
    all_results.append(r)

    # Critical signal check (what determines survival in THIS industry)
    cs = r['idea_signal_data']
    print(f"   Regime:          {r['regime']}")
    print(f"   TAS:             {r['tas']:.4f}  [{r['signal_tier']}]  -> {r['decision']}")
    print(f"   Macro cap:       {r['macro_cap_applied']:.0%}  (macro capped at this level for {idea['sector']})")
    print(f"   Critical signal: {cs['critical_signal_name']} = {cs['critical_signal_score']:.3f}")
    print(f"   Dominant risk:   {cs['dominant_risk']}")
    print(f"   Failure mode:    {cs['industry_failure_mode']}")
    print(f"   Moat type:       {cs['industry_moat_type']}")
    print(f"   Sanity:          {'VALID' if r['sanity']['valid'] else 'ERRORS: ' + str(r['sanity']['errors'])}")
    if r['sanity']['warnings']:
        print(f"   warnings:        {r['sanity']['warnings']}")

# Final variability check across all 4 ideas
var = validate_signal_variability(all_results)
print(f"\n{'='*72}")
print(f"VARIABILITY CHECK (should be >= 0.08 spread):")
print(f"  Spread: {var['spread']:.4f}  |  {'PASSED' if var['passed'] else 'FAILED'}")
print(f"  Signals: {var['signals']}")
if var['warning']:
    print(f"    {var['warning']}")

print(f"\n[KEY RESULT] Different industries -> different critical signals -> different logic:")
for i, (idea, res) in enumerate(zip(test_ideas, all_results)):
    cs = res['idea_signal_data']
    print(f"  {idea['sector'].upper():12} | {cs['critical_signal_name']:22} | TAS={res['tas']:.3f} [{res['signal_tier']}]")


## 13. Agent A1 — NLP Parser (LLM with Temperature=0)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AGENT A1 — NLP Parser (Groq LLM, Temperature=0)
# Free tier: 14,400 requests/day — console.groq.com
# ═══════════════════════════════════════════════════════════════
!pip install groq -q

import os, json, re
from groq import Groq

# ── Constants ─────────────────────────────────────────────────
VALID_SECTORS = [
    'fintech', 'ecommerce', 'healthtech', 'edtech',
    'saas', 'logistics', 'agritech', 'other'
]
VALID_COUNTRIES = [
    'EG','SA','AE','US','GB','DE','FR','IN','PK',
    'NG','KE','MA','TN','JO','LB','KW','QA','TR','ID','MY'
]

KEYWORD_FALLBACK_SECTORS = {
    'fintech':    ['finance','payment','fintech','bank','loan',
                   'lending','invoice','insurance','wallet','money'],
    'ecommerce':  ['ecommerce','e-commerce','shop','store',
                   'retail','marketplace','delivery','commerce'],
    'healthtech': ['health','medical','doctor','clinic',
                   'hospital','pharma','biotech','mental'],
    'edtech':     ['education','learning','school','university',
                   'course','tutor','edtech','training'],
    'saas':       ['saas','software','platform','dashboard',
                   'tool','api','enterprise','cloud','b2b','crm'],
    'logistics':  ['logistics','shipping','supply chain',
                   'warehouse','transport','fleet','trucking'],
    'agritech':   ['agri','farm','crop','harvest',
                   'food','agriculture','irrigation'],
}
KEYWORD_FALLBACK_COUNTRIES = {
    'EG': ['egypt','cairo','egyptian','مصر','القاهرة'],
    'SA': ['saudi','ksa','riyadh','jeddah','السعودية'],
    'AE': ['uae','dubai','abu dhabi','emirates','الإمارات'],
    'MA': ['morocco','casablanca','المغرب'],
    'NG': ['nigeria','lagos','abuja'],
    'KE': ['kenya','nairobi'],
    'TN': ['tunisia','tunis','تونس'],
    'JO': ['jordan','amman','الأردن'],
}

# ── Helper: parse + validate LLM response ─────────────────────
def parse_llm_response(raw: str) -> dict | None:
    try:
        data = json.loads(raw.strip())
    except json.JSONDecodeError:
        match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        if not match: return None
        try: data = json.loads(match.group())
        except: return None

    sector  = str(data.get('sector',  '')).lower().strip()
    country = str(data.get('country', '')).upper().strip()
    if sector  not in VALID_SECTORS:   sector  = 'other'
    if country not in VALID_COUNTRIES: country = 'EG'
    return {'sector': sector, 'country': country}

# ── Helper: keyword fallback (no API needed) ──────────────────
def keyword_fallback(idea: str) -> dict:
    text = idea.lower()
    sector = 'other'
    for sec, kws in KEYWORD_FALLBACK_SECTORS.items():
        if any(k in text for k in kws):
            sector = sec
            break
    country = 'EG'
    for code, kws in KEYWORD_FALLBACK_COUNTRIES.items():
        if any(k in text for k in kws):
            country = code
            break
    return {'sector': sector, 'country': country}

# ── Main Agent A1 function ────────────────────────────────────
def agent_a1_parse(idea: str, groq_client=None) -> dict:
    """
    Parse startup idea into {sector, country}.
    Uses Groq LLM (Temperature=0) if client provided,
    falls back to keyword matching automatically.
    """
    SYSTEM_PROMPT = """You are a startup idea parser. Extract ONLY the sector and country.
Reply with EXACTLY this JSON and nothing else:
{"sector": "...", "country": "ISO2_CODE"}

Valid sectors: fintech, ecommerce, healthtech, edtech, saas, logistics, agritech, other
Valid country codes: EG, SA, AE, US, GB, DE, FR, IN, PK, NG, KE, MA, TN, JO, LB, KW, QA, TR
Default country = EG if not mentioned in the idea."""

    if groq_client is not None:
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": idea},
                ],
                temperature=0.0,
                max_tokens=60,
                response_format={"type": "json_object"},
            )
            raw    = response.choices[0].message.content
            parsed = parse_llm_response(raw)
            if parsed:
                print(f"[A1] LLM result: {parsed}")
                return parsed
            print(f"[A1] LLM returned unparseable output, using fallback")
        except Exception as e:
            print(f"[A1] LLM error ({e}), using keyword fallback")

    result = keyword_fallback(idea)
    print(f"[A1] Keyword fallback result: {result}")
    return result

# ── Setup Groq client ─────────────────────────────────────────
# Paste your Groq key here (get free key from console.groq.com)
GROQ_API_KEY = "your_groq_key_here"

if GROQ_API_KEY != "your_groq_key_here":
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client ready.")
else:
    groq_client = None
    print("No API key set — using keyword fallback mode.")

# ── TEST ──────────────────────────────────────────────────────
test_ideas = [
    "Invoice financing app for Egyptian SMEs",
    "E-commerce last-mile delivery platform in Cairo",
    "Mental health app for Saudi Arabia",
    "SaaS CRM for logistics companies in UAE",
    "Agricultural irrigation system for Morocco farmers",
]

print("\n" + "="*55)
print("Agent A1 — Startup Idea Parser")
print("="*55)
for idea in test_ideas:
    result = agent_a1_parse(idea, groq_client)
    print(f"Idea:   {idea}")
    print(f"Output: sector={result['sector']} | country={result['country']}")
    print()


## 14. Build Context Files for Agents A2 & A4

In [ ]:
# ── Unicorn Competitors (Agent A2) ────────────────────────────
unicorns = pd.read_csv('unicorn_startup_companies.csv', encoding='utf-8', on_bad_lines='skip')
print(f"Unicorns: {unicorns.shape}")
print(unicorns.head(3))

def map_unicorn_sector(industry_str):
    if pd.isna(industry_str): return 'other'
    text = str(industry_str).lower()
    for sector, keywords in SECTOR_MAP.items():
        if any(k.lower() in text for k in keywords):
            return sector
    return 'other'

unicorns['sector'] = unicorns['Industry'].apply(map_unicorn_sector)

competitors_by_sector = {}
for sec in unicorns['sector'].unique():
    sub = unicorns[unicorns['sector'] == sec][['Company', 'Valuation ($B)', 'Country', 'Industry']].head(10)
    competitors_by_sector[sec] = sub.to_dict(orient='records')

with open(f'{MODELS_DIR}/competitors_context.json', 'w') as f:
    json.dump(competitors_by_sector, f, indent=2, default=str)

print("\nCompetitors by sector:")
for sec, comps in competitors_by_sector.items():
    print(f"  {sec}: {len(comps)} unicorn competitors")
print("Saved: competitors_context.json")

# ── Sentiment / Demand Data (Agent A4) ────────────────────────
try:
    sentiment = pd.read_csv('all-data.csv', encoding='latin-1',
                            names=['sentiment', 'text'], on_bad_lines='skip')
    print(f"\nSentiment data: {sentiment.shape}")
    print(sentiment['sentiment'].value_counts())

    # Save as context
    sentiment_sample = sentiment.sample(min(200, len(sentiment)), random_state=42)
    with open(f'{MODELS_DIR}/sentiment_context.json', 'w') as f:
        json.dump(sentiment_sample.to_dict(orient='records'), f, indent=2, default=str)
    print("Saved: sentiment_context.json")
except Exception as e:
    print(f"Sentiment data issue: {e}")


## 15. Download All Models (for Streamlit Dashboard)

In [ ]:
import zipfile
from google.colab import files

# ── List all saved models ─────────────────────────────────────
saved_files = os.listdir(MODELS_DIR)
print("Files in models/ directory:")
for f in sorted(saved_files):
    size = os.path.getsize(f'{MODELS_DIR}/{f}')
    print(f"  {f:<40} {size/1024:.1f} KB")

# ── Zip everything ────────────────────────────────────────────
zip_path = 'midan_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in saved_files:
        zf.write(f'{MODELS_DIR}/{f}', f)

zip_size = os.path.getsize(zip_path) / (1024*1024)
print(f"\nZipped all models → {zip_path} ({zip_size:.1f} MB)")

# ── Download ──────────────────────────────────────────────────
files.download(zip_path)
print("Download started!")


## Pipeline Complete

### Models Saved
| File | Purpose |
|---|---|
| `svm_global.pkl` | Global SVM classifier (fallback) |
| `svm_fintech.pkl` etc | Sector-specific SVMs |
| `lgb_surrogate.pkl` | LightGBM mirrors SVM for SHAP |
| `scaler_global.pkl` | StandardScaler for X_new |
| `pca_global.pkl` | PCA 2D for visualization |
| `label_encoder.pkl` | Regime name ↔ integer mapping |
| `sarima_*.pkl` | SARIMA per sector |
| `cluster_names.json` | Deterministic cluster labels |
| `sarima_results.json` | 90-day forecasts + drift flags |
| `competitors_context.json` | Agent A2 competitor data |
| `sentiment_context.json` | Agent A4 demand signals |
| `fit_table_heatmap.png` | Regime × Business Model fit visualization |
| `idea_signal_comparison.png` | Variability proof: two ideas, same regime |

### Architecture Summary (v2.0)

| Layer | Role | Output |
|---|---|---|
| **L1** LLM Parser | Extracts 8 idea features from raw text | `business_model`, `target_segment`, `differentiation_score`, … |
| **L2** ML Pipeline | Classifies market regime from 5 macro features | `regime`, `confidence`, `shap_dict`, `sarima_trend` |
| **L3** Idea Scorer | Computes idea-specific signal using regime×fit table | `idea_signal` (0.12–0.95), `signal_tier` |
| **L4** Composite TAS | Merges all layers | `tas = conf×0.30 + sarima×0.20 + idea_signal×0.35 + xai×0.15` |

### Next Step
Place `midan_models.zip` → extract into `models/` folder in the FastAPI project.
The `api.py` backend will auto-load all `.pkl` files.
The `midan.html` frontend will display `combined_signals` as "Market Signals" + "Idea Signals".